# 02. EDA & Statistics — TARGET과의 연관성

> 이 노트북은 **설명용**이다. 계산 로직은 모두 `src/credit_risk/eda.py`, `src/credit_risk/stats.py`에 있으며, 보고서는 `python -m credit_risk.cli eda`로 재현한다.
>
> 용어: TARGET 평균 = **관측 상환곤란율 (Observed Difficulty Rate)**, AMT_CREDIT = **Exposure Proxy (AMT_CREDIT)**. 모든 관계는 연관이며 인과가 아니다. 합성 데이터가 적재된 경우 결과는 해석에 사용하지 않는다.

대표본에서는 작은 차이도 유의해지므로 **효과크기**(rank-biserial r, Cramér's V)로 판단하고, 다중 비교는 Benjamini–Hochberg로 보정한다 (Bonferroni 병기).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("DATA_DIR", "./data/raw")  # only used for the data directory; DB creds come from .env

import pandas as pd
from credit_risk.config import get_settings
from credit_risk.db import connect
from credit_risk import eda, stats

settings = get_settings()
pd.set_option("display.max_columns", 50)

## 전체 파이프라인 실행 (보고서·그림·JSON 생성)

In [ ]:
payload = eda.run_eda(settings)
pd.DataFrame(payload['insights'])[['id', 'title', 'metric', 'value', 'powerbi_page']]

## 개별 함수로 다시 보기 — 고객 단위 프레임

In [ ]:
with connect(settings) as conn:
    df = eda.load_train_frame(conn, ['target', 'amt_credit', 'amt_income_total', 'ext_source_mean',
                                     'age_years', 'inst_late_ratio', 'name_income_type'])
eda.target_balance(df)

## 수치형: Mann–Whitney U + rank-biserial r

In [ ]:
tests = eda.target_comparison(df, ['amt_income_total', 'ext_source_mean', 'inst_late_ratio'], 'demo')
tests[['feature', 'median1', 'median0', 'p_value', 'effect_size', 'effect_label']]

## 구간별 관측 상환곤란율과 Wilson 95% CI

In [ ]:
b = eda.binned_risk(df, eda.quantile_band(df['ext_source_mean'], 10, 'D'), 'ext_source_decile', 'ext_source_mean')
b['test'], b['table']

## 범주형: Chi-square + Cramér's V, 다중 비교 보정

In [ ]:
cat = eda.categorical_tests(df, ['name_income_type'], 'demo')
stats.adjust_results(pd.concat([tests, cat]))[['feature', 'p_value', 'p_adj_bh', 'p_adj_bonferroni', 'effect_size']]

## 공정성 고지
연령·성별·가족 속성은 기술통계로만 보며 승인 기준으로 사용하지 않는다. 생성된 `reports/eda_report.md`, `reports/statistical_analysis.md`를 참고한다.